In [3]:
import operator
from typing import TypedDict, Literal, Optional, Annotated
from pydantic import BaseModel, Field

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END


# ==========================================
# 1. Pydantic Schemas for Structured Output
# ==========================================

class TweetEvaluationSchema(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(
        description="Whether the tweet meets all quality criteria or needs improvement."
    )
    feedback: str = Field(
        description="Constructive feedback explaining the reasoning behind the evaluation."
    )


# ==========================================
# 2. State Definition
# ==========================================

class TweetState(TypedDict):
    topic: str
    tweet: Optional[str]
    evaluation: Optional[str]
    feedback: Optional[str]
    iteration: int
    max_iteration: int
    tweet_history: Annotated[list[str], operator.add]
    feedback_history: Annotated[list[str], operator.add]


# ==========================================
# 3. Model Initialization & Node Functions
# ==========================================

# Initialize LLMs (Evaluator uses structured output)
generator_llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0.7)
evaluator_llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0.0)
optimizer_llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0.7)

structured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluationSchema)


def generate_tweet(state: TweetState) -> dict:
    """Generates the initial candidate tweet."""
    messages = [
        SystemMessage(content="You are a funny and clever Twitter influencer."),
        HumanMessage(content=(
            f"Write a short, original, and hilarious tweet on the topic: '{state['topic']}'.\n"
            "Rules: Do NOT use question-answer format. Stay under 280 characters. "
            "Use observational humor, irony, or sarcasm."
        ))
    ]
    res = generator_llm.invoke(messages)
    tweet_text = res.content
    return {
        "tweet": tweet_text,
        "tweet_history": [tweet_text]
    }


def evaluate_tweet(state: TweetState) -> dict:
    """Evaluates the candidate tweet using structured Pydantic output."""
    messages = [
        SystemMessage(content=(
            "You are a ruthless, no-nonsense Twitter critic. "
            "Evaluate tweets based on humor, originality, virality, and formatting."
        )),
        HumanMessage(content=(
            f"Evaluate the following tweet:\n'{state['tweet']}'\n\n"
            "Rules: Reject immediately if it uses Q&A style, exceeds 280 characters, "
            "or uses generic jokes."
        ))
    ]
    res = structured_evaluator_llm.invoke(messages)
    return {
        "evaluation": res.evaluation,
        "feedback": res.feedback,
        "feedback_history": [res.feedback]
    }


def optimize_tweet(state: TweetState) -> dict:
    """Refines the tweet based on the evaluator's specific feedback."""
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on feedback."),
        HumanMessage(content=(
            f"Improve the tweet based on this feedback:\n'{state['feedback']}'\n"
            f"Topic: '{state['topic']}'\n"
            f"Original Tweet: '{state['tweet']}'\n\n"
            "Rewrite it as a viral-worthy tweet under 280 characters. Avoid Q&A style."
        ))
    ]
    res = optimizer_llm.invoke(messages)
    new_tweet = res.content
    return {
        "tweet": new_tweet,
        "iteration": state["iteration"] + 1,
        "tweet_history": [new_tweet]
    }


# ==========================================
# 4. Conditional Routing Function
# ==========================================

def route_evaluation(state: TweetState) -> str:
    """Routes execution based on evaluation output and max iteration cap."""
    if state["evaluation"] == "approved" or state["iteration"] >= state["max_iteration"]:
        return "approved"
    else:
        return "needs_improvement"


# ==========================================
# 5. Graph Assembly & Compilation
# ==========================================

graph = StateGraph(TweetState)

# Add Nodes
graph.add_node("generate", generate_tweet)
graph.add_node("evaluate", evaluate_tweet)
graph.add_node("optimize", optimize_tweet)

# Add Edges
graph.add_edge(START, "generate")
graph.add_edge("generate", "evaluate")

# Add Conditional Routing Edge
graph.add_conditional_edges(
    "evaluate",
    route_evaluation,
    {
        "approved": END,
        "needs_improvement": "optimize"
    }
)

# Loop Edge: Refined draft goes back for evaluation
graph.add_edge("optimize", "evaluate")

# Compile Application Graph
app = graph.compile()


# ==========================================
# 6. Execution & Verification
# ==========================================

if __name__ == "__main__":
    initial_state = {
        "topic": "Indian Railways",
        "iteration": 1,
        "max_iteration": 5,
        "tweet_history": [],
        "feedback_history": []
    }

    result = app.invoke(initial_state)

    print("==========================================")
    print("         FINAL WORKFLOW RESULT            ")
    print("==========================================")
    print(f"Total Iterations: {result['iteration']}")
    print(f"Final Status:     {result['evaluation']}")
    print(f"\nFinal Tweet:\n{result['tweet']}\n")

    print("==========================================")
    print("       TWEET HISTORY EVOLUTION            ")
    print("==========================================")
    for idx, (tweet, feedback) in enumerate(zip(result['tweet_history'], result['feedback_history']), 1):
        print(f"\n[Iteration {idx}]")
        print(f"Tweet Draft: {tweet}")
        print(f"Feedback:    {feedback}")

d:\anaconda3\envs\lgenv\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
d:\anaconda3\envs\lgenv\Lib\site-packages\langchain_google_genai\chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


         FINAL WORKFLOW RESULT            
Total Iterations: 1
Final Status:     approved

Final Tweet:
[{'type': 'text', 'text': 'Indian Railways is the only place on earth where buying a confirmed ticket is just an invitation to play an extreme psychological thriller called "Will My Seat Actually Be Mine?" starring three uncles, a family of six, and a goat.', 'extras': {'signature': 'EmAKXgFpFH0T6BGyb+zXfA++hGatpNP4Dshjl4/uTWS9ygGFK5XVCYrji2hqCIsEp4fYmhDhyxZp8TJwHX7oPaXfZsheqEDVsPPhXZDSd6b02ZsSeyQ51sbRqHlRNMveKUI='}}]

       TWEET HISTORY EVOLUTION            

[Iteration 1]
Tweet Draft: [{'type': 'text', 'text': 'Indian Railways is the only place on earth where buying a confirmed ticket is just an invitation to play an extreme psychological thriller called "Will My Seat Actually Be Mine?" starring three uncles, a family of six, and a goat.', 'extras': {'signature': 'EmAKXgFpFH0T6BGyb+zXfA++hGatpNP4Dshjl4/uTWS9ygGFK5XVCYrji2hqCIsEp4fYmhDhyxZp8TJwHX7oPaXfZsheqEDVsPPhXZDSd6b02ZsSeyQ51